
### Week 11: Microsoft Azure AI Services - Vision, Security, Anomaly Detection & Containerization



### 1. Environment Setup & Azure SDK Configurations


In [ ]:
import os
import json
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw, ImageFont
import io

print("Azure AI Services Practical Lab initialized.")
print("SDK components & utility dependencies ready.")


Azure AI Services Practical Lab initialized.
SDK components & utility dependencies ready.


---
## Module 1 & 2: Azure AI Service Selection, Security & Responsible AI
### Core Architectural Guidelines:
1. **Vision Solutions:** Azure Computer Vision (general analysis, OCR), Custom Vision (domain-specific classes), Face API.
2. **Language Solutions:** Azure AI Language (Sentiment, NER, Summarization), Azure Translator.
3. **Decision Solutions:** Azure Anomaly Detector (time-series anomalies), Content Moderator (toxic text/media filter), Personalizer (reinforcement learning recommendation).
4. **Speech Solutions:** Azure Speech (Speech-to-Text, Text-to-Speech, Speech Translation).

### Security Architecture:
- Key rotation via Azure Key Vault
- Private Endpoints & Virtual Network (VNet) service tagging
- Managed Identities (Microsoft Entra ID / RBAC) eliminating hardcoded keys.


In [ ]:
# Simulating Azure AI Configuration and Authentication Management
class AzureAIServiceConfig:
    def __init__(self, endpoint=None, api_key=None, region="eastus"):
        # Reads from environment variables or defaults to placeholder credentials
        self.endpoint = endpoint or os.getenv("AZURE_AI_ENDPOINT", "https://eastus.api.cognitive.microsoft.com/")
        self.api_key = api_key or os.getenv("AZURE_AI_KEY", "SAMPLE_AZURE_AI_KEY_REDACTED")
        self.region = region

    def get_auth_headers(self):
        return {
            "Ocp-Apim-Subscription-Key": self.api_key,
            "Content-Type": "application/json"
        }

    def validate_responsible_ai_checklist(self):
        checklist = {
            "Fairness": "Model evaluated across diverse demographic subgroups.",
            "Reliability & Safety": "Tested under adversarial inputs and edge cases.",
            "Privacy & Security": "PII redacted; data encrypted in-transit (TLS 1.3) and at-rest.",
            "Inclusiveness": "Accessible UI and support for multi-language inputs.",
            "Transparency": "Model cards, confidence scores, and limitations documented.",
            "Accountability": "Human-in-the-loop audit logs enabled."
        }
        return checklist

azure_config = AzureAIServiceConfig()
print("Azure Service Endpoint:", azure_config.endpoint)
print("Security Status       : Headers Configured")
print("\n--- Responsible AI Governance Assessment ---")
for principle, status in azure_config.validate_responsible_ai_checklist().items():
    print(f"  [x] {principle:<22}: {status}")


Azure Service Endpoint: https://eastus.api.cognitive.microsoft.com/
Security Status       : Headers Configured

--- Responsible AI Governance Assessment ---
  [x] Fairness              : Model evaluated across diverse demographic subgroups.
  [x] Reliability & Safety  : Tested under adversarial inputs and edge cases.
  [x] Privacy & Security    : PII redacted; data encrypted in-transit (TLS 1.3) and at-rest.
  [x] Inclusiveness         : Accessible UI and support for multi-language inputs.
  [x] Transparency          : Model cards, confidence scores, and limitations documented.
  [x] Accountability        : Human-in-the-loop audit logs enabled.


---
## Module 3: Azure Computer Vision - Image Analysis
Calling the Computer Vision API to extract:
- Visual categories and automated caption descriptions
- Confidence ratings
- Object tags and localized bounding boxes.


In [ ]:
# Azure Computer Vision Image Analysis Client Implementation
class AzureComputerVisionClient:
    def __init__(self, config):
        self.config = config

    def analyze_image(self, image_url, visual_features=['Categories', 'Description', 'Color', 'Tags', 'Objects']):
        # Standard Azure Computer Vision v3.2 / v4.0 endpoint
        endpoint_url = f"{self.config.endpoint.rstrip('/')}/vision/v3.2/analyze"
        params = {"visualFeatures": ",".join(visual_features), "language": "en"}
        payload = {"url": image_url}

        # Simulating parsed API response conforming directly to Azure Computer Vision schema
        simulated_response = {
            "categories": [{"name": "outdoor_mountain", "score": 0.982}],
            "description": {
                "tags": ["mountain", "snow", "outdoor", "sky", "hiking", "nature", "scenic"],
                "captions": [{
                    "text": "A snowy mountain range under a clear blue sky with hikers in foreground",
                    "confidence": 0.9412
                }]
            },
            "objects": [
                {"object": "Person", "confidence": 0.884, "rectangle": {"x": 120, "y": 320, "w": 65, "h": 140}},
                {"object": "Backpack", "confidence": 0.812, "rectangle": {"x": 145, "y": 350, "w": 40, "h": 55}},
                {"object": "Person", "confidence": 0.912, "rectangle": {"x": 230, "y": 310, "w": 70, "h": 150}}
            ],
            "color": {"dominantColorForeground": "White", "dominantColorBackground": "Blue", "isBWImg": False}
        }
        return simulated_response

cv_client = AzureComputerVisionClient(azure_config)
sample_image_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-sample-data-files/master/ComputerVision/Images/mountain.jpg"
vision_result = cv_client.analyze_image(sample_image_url)

print("=== Azure Computer Vision Analysis Output ===")
print("Generated Caption :", vision_result['description']['captions'][0]['text'])
print(f"Caption Confidence: {vision_result['description']['captions'][0]['confidence'] * 100:.2f}%")
print("Top Extracted Tags:", vision_result['description']['tags'][:5])
print(f"Detected Objects  : {len(vision_result['objects'])} objects identified")
for obj in vision_result['objects']:
    rect = obj['rectangle']
    print(f"  - {obj['object']:<10} (Confidence: {obj['confidence']*100:.1f}%) at [x:{rect['x']}, y:{rect['y']}, w:{rect['w']}, h:{rect['h']}]")


=== Azure Computer Vision Analysis Output ===
Generated Caption : A snowy mountain range under a clear blue sky with hikers in foreground
Caption Confidence: 94.12%
Top Extracted Tags: ['mountain', 'snow', 'outdoor', 'sky', 'hiking']
Detected Objects  : 3 objects identified
  - Person     (Confidence: 88.4%) at [x:120, y:320, w:65, h:140]
  - Backpack   (Confidence: 81.2%) at [x:145, y:350, w:40, h:55]
  - Person     (Confidence: 91.2%) at [x:230, y:310, w:70, h:150]


In [ ]:
# Visualizing Bounding Boxes on Simulated Image Canvas
fig, ax = plt.subplots(figsize=(8, 5))
# Create synthetic image background
canvas = np.zeros((500, 600, 3), dtype=np.uint8)
canvas[:250, :] = [135, 206, 235]  # Sky
canvas[250:, :] = [240, 248, 255]  # Snow
ax.imshow(canvas)

# Overlay Azure Vision bounding boxes
for obj in vision_result['objects']:
    r = obj['rectangle']
    box = patches.Rectangle((r['x'], r['y']), r['w'], r['h'], linewidth=2, edgecolor='crimson', facecolor='none')
    ax.add_patch(box)
    ax.text(r['x'], r['y'] - 8, f"{obj['object']} ({obj['confidence']*100:.0f}%)",
            color='white', backgroundcolor='crimson', fontsize=9, weight='bold')

plt.title('Azure Computer Vision: Localized Object Detection Output')
plt.axis('off')
plt.show()


---
## Module 4: Optical Character Recognition (OCR) with Azure Read API
The Azure Read API uses deep learning models to extract printed and handwritten text from images, invoices, and PDF documents.


In [ ]:
class AzureReadAPIClient:
    def __init__(self, config):
        self.config = config

    def extract_text(self, document_url):
        # Simulated two-step Azure Read API response (POST operation-location -> GET read results)
        simulated_read_output = {
            "status": "succeeded",
            "recognitionResults": [
                {
                    "page": 1,
                    "angle": 0.45,
                    "width": 800,
                    "height": 600,
                    "unit": "pixel",
                    "lines": [
                        {"text": "INVOICE #INV-2024-0891", "confidence": 0.998, "boundingBox": [50, 40, 260, 40, 260, 65, 50, 65]},
                        {"text": "Billed To: National Training Academy", "confidence": 0.985, "boundingBox": [50, 90, 380, 90, 380, 115, 50, 115]},
                        {"text": "Item 1: Azure Cloud GPU Cluster - $1,250.00", "confidence": 0.992, "boundingBox": [50, 150, 450, 150, 450, 175, 50, 175]},
                        {"text": "Item 2: Custom Vision Storage Tier - $150.00", "confidence": 0.989, "boundingBox": [50, 190, 440, 190, 440, 215, 50, 215]},
                        {"text": "Total Due: $1,400.00", "confidence": 0.999, "boundingBox": [50, 260, 220, 260, 220, 290, 50, 290]}
                    ]
                }
            ]
        }
        return simulated_read_output

read_client = AzureReadAPIClient(azure_config)
doc_results = read_client.extract_text("https://example.com/invoice_sample.png")

print(f"Azure Read API Status: {doc_results['status'].upper()}")
print(f"Recognized Document Lines (Page 1):\n")
print(f"{'Line #':<8} | {'Confidence':<12} | {'Recognized Text'}")
print("-" * 65)
for idx, line in enumerate(doc_results['recognitionResults'][0]['lines'], 1):
    print(f"{idx:<8} | {line['confidence']*100:<11.2f}% | {line['text']}")


Azure Read API Status: SUCCEEDED
Recognized Document Lines (Page 1):

Line #   | Confidence   | Recognized Text
-----------------------------------------------------------------
1        | 99.80%       | INVOICE #INV-2024-0891
2        | 98.50%       | Billed To: National Training Academy
3        | 99.20%       | Item 1: Azure Cloud GPU Cluster - $1,250.00
4        | 98.90%       | Item 2: Custom Vision Storage Tier - $150.00
5        | 99.90%       | Total Due: $1,400.00


---
## Module 5: Azure Custom Vision Model Workflow
Overview and execution of Custom Vision service:
1. Create Training Project with Domain type (e.g., General, Retail, Compact)
2. Upload and tag image training sets
3. Train model iteration (Fast Training vs Advanced Training)
4. Evaluate metrics: Precision, Recall, and mAP (Mean Average Precision)
5. Publish model iteration to Prediction API endpoint.


In [ ]:
# Simulating Azure Custom Vision Project Lifecycle
class AzureCustomVisionManager:
    def __init__(self, project_name="SafetyGearDetector"):
        self.project_name = project_name
        self.tags = ["Helmet", "SafetyVest", "Gloves"]
        self.iteration = 1

    def train_iteration(self):
        print(f"Training iteration #{self.iteration} for project '{self.project_name}'...")
        time.sleep(1) # simulate training step
        metrics = {
            "Precision": 0.942,
            "Recall": 0.918,
            "mAP (Mean Average Precision)": 0.935,
            "Images Trained": 250
        }
        return metrics

    def predict_image(self, image_data):
        return [
            {"tag": "Helmet", "probability": 0.965, "boundingBox": {"left": 0.25, "top": 0.1, "width": 0.2, "height": 0.2}},
            {"tag": "SafetyVest", "probability": 0.923, "boundingBox": {"left": 0.22, "top": 0.32, "width": 0.35, "height": 0.45}}
        ]

cv_mgr = AzureCustomVisionManager()
metrics = cv_mgr.train_iteration()

print("\nIteration #1 Training Complete. Model Metrics:")
for k, v in metrics.items():
    print(f"  {k:<30}: {v}")

predictions = cv_mgr.predict_image("site_worker.jpg")
print("\nPublished Endpoint Inference Test:")
for p in predictions:
    print(f"  Detected Tag: {p['tag']:<12} | Probability: {p['probability']*100:.2f}%")


Training iteration #1 for project 'SafetyGearDetector'...

Iteration #1 Training Complete. Model Metrics:
  Precision                     : 0.942
  Recall                        : 0.918
  mAP (Mean Average Precision)  : 0.935
  Images Trained                : 250

Published Endpoint Inference Test:
  Detected Tag: Helmet       | Probability: 96.50%
  Detected Tag: SafetyVest   | Probability: 92.30%


---
## Module 6: Azure Anomaly Detector & Content Moderator
1. **Anomaly Detector:** Evaluates time-series data to detect anomalous outliers and trend shifts.
2. **Content Moderator:** Evaluates text for profanity, sensitive data (PII: emails, phone numbers), and offensive categories.


In [ ]:
# Part 1: Azure Anomaly Detector Demonstration
# Generating sample time series with intentional anomaly spikes
dates = pd.date_range("2024-01-01", periods=30, freq="D")
values = 100 + 10 * np.sin(np.linspace(0, 3*np.pi, 30)) + np.random.normal(0, 2, 30)
# Injecting anomalies
values[12] = 165.0  # Spike anomaly
values[24] = 45.0   # Dip anomaly

df_metrics = pd.DataFrame({"timestamp": dates, "value": values})

# Simulating Anomaly Detector entire detect API output
mean_val = df_metrics['value'].mean()
std_val = df_metrics['value'].std()
df_metrics['is_anomaly'] = np.abs(df_metrics['value'] - mean_val) > (2.2 * std_val)

anomalies = df_metrics[df_metrics['is_anomaly']]
print(f"Total Data Points Evaluated: {len(df_metrics)}")
print(f"Anomalies Detected         : {len(anomalies)}\n")
print("Detected Anomaly Timestamps:")
print(anomalies[['timestamp', 'value']].to_string(index=False))


Total Data Points Evaluated: 30
Anomalies Detected         : 2

Detected Anomaly Timestamps:
 timestamp      value
2024-01-13 165.000000
2024-01-25  45.000000


In [ ]:
# Part 2: Azure Content Moderator Demonstration
sample_user_comment = "Call me at +1-425-555-0199 or email user_test@corp.com. This service was completely awful!"

def azure_content_moderation(text):
    # Simulates Content Moderator screening for PII and toxicity
    import re
    emails = re.findall(r'[\w\.-]+@[\w\.-]+', text)
    phones = re.findall(r'\+?\d[\d -]{8,14}\d', text)

    # Toxicity / Sentiment screening scores
    return {
        "Original Text": text,
        "PII Detected": {
            "Emails": emails,
            "Phone Numbers": phones
        },
        "Classification": {
            "Category1 (Explicit Content)": 0.02,
            "Category2 (Suggestive Content)": 0.04,
            "Category3 (Offensive Language)": 0.72
        },
        "Moderation Action": "FLAG_FOR_REVIEW" if (emails or phones) else "APPROVE"
    }

mod_result = azure_content_moderation(sample_user_comment)
print("=== Azure Content Moderator Inspection Report ===")
print("Inbound Content  :", mod_result["Original Text"])
print("Moderation Action:", mod_result["Moderation Action"])
print("Detected PII     :", mod_result["PII Detected"])
print("Category Scores  :", mod_result["Classification"])


=== Azure Content Moderator Inspection Report ===
Inbound Content  : Call me at +1-425-555-0199 or email user_test@corp.com. This service was completely awful!
Moderation Action: FLAG_FOR_REVIEW
Detected PII     : {'Emails': ['user_test@corp.com'], 'Phone Numbers': ['+1-425-555-0199']}
Category Scores  : {'Category1 (Explicit Content)': 0.02, 'Category2 (Suggestive Content)': 0.04, 'Category3 (Offensive Language)': 0.72}


---
## Module 7: Azure Cognitive Services Prebuilt Container Deployment
To meet low-latency, privacy, and disconnected/on-premises requirements, Azure allows running prebuilt containerized models locally using Docker.


In [ ]:
# Documenting Container Deployment Commands for Azure Read & Computer Vision
docker_deployment_commands = '''
# 1. Pull Azure Computer Vision Read OCR prebuilt Docker image
docker pull mcr.microsoft.com/azure-cognitive-services/vision/read:3.2-stt

# 2. Run Container locally, mounting local memory and binding billing endpoint
docker run --rm -it -p 5000:5000 --memory 16g --cpus 4 \
  mcr.microsoft.com/azure-cognitive-services/vision/read:3.2-stt \
  Eula=accept \
  Billing=https://eastus.api.cognitive.microsoft.com/ \
  ApiKey=YOUR_AZURE_KEY_HERE

# 3. Test Local Container Health Endpoint
curl -v -X GET "http://localhost:5000/status"
'''

print("Container Deployment Script & CLI Reference:")
print(docker_deployment_commands)


Container Deployment Script & CLI Reference:

# 1. Pull Azure Computer Vision Read OCR prebuilt Docker image
docker pull mcr.microsoft.com/azure-cognitive-services/vision/read:3.2-stt

# 2. Run Container locally, mounting local memory and binding billing endpoint
docker run --rm -it -p 5000:5000 --memory 16g --cpus 4 \
  mcr.microsoft.com/azure-cognitive-services/vision/read:3.2-stt \
  Eula=accept \
  Billing=https://eastus.api.cognitive.microsoft.com/ \
  ApiKey=YOUR_AZURE_KEY_HERE

# 3. Test Local Container Health Endpoint
curl -v -X GET "http://localhost:5000/status"


### Summary & Key Azure AI Learnings:
- **Service Selection:** Knowing when to use prebuilt services (Computer Vision, Read OCR) vs customized endpoints (Custom Vision) speeds up enterprise AI delivery.
- **Security & Ethics:** Responsible AI principles (fairness, transparency, data privacy) and Azure RBAC / Key Vault credentials protect customer datasets.
- **Applied AI:** Automated Anomaly Detection and Content Moderation provide real-time operational safety and quality assurance.
- **Edge Deployment:** Docker container images enable running Cognitive Services in air-gapped or on-premises networks with consistent API schemas.
